# Лабораторная работа №5
## Ансамблевые методы: бэггинг, AdaBoost и градиентный бустинг

**Тема:** Случайный лес и сверхслучайные деревья (бэггинг), AdaBoost и градиентный бустинг

**Цель работы:** Обучить и сравнить ансамблевые модели на задаче классификации по одной выбранной метрике

**Выполнил:** [ФИО студента]  
**Группа:** [Номер группы]  
**Дата:** [Дата выполнения]


## 1. Описание задания

### Задачи лабораторной работы
1. Выбрать набор данных для задачи классификации или регрессии.
2. При необходимости обработать пропуски и закодировать категориальные признаки.
3. Разделить выборку на обучающую и тестовую с помощью `train_test_split`.
4. Обучить **две** модели группы бэггинга (в работе: **случайный лес** и **сверхслучайные деревья**).
5. Обучить **AdaBoost** и **градиентный бустинг** (`GradientBoostingClassifier`).
6. Оценить качество **одной** подходящей метрикой и сравнить модели.

Используется датасет **Titanic** (бинарная классификация). Для деревянных ансамблей масштабирование признаков не обязательно — после кодирования признаки подаются в модели как есть.


## 2. Импорт библиотек


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
import warnings

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
)
from sklearn.metrics import accuracy_score

warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8-darkgrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 10

print("Библиотеки успешно импортированы")


## 3. Загрузка и предобработка данных

Удаляем неинформативные столбцы, заполняем пропуски, применяем one-hot кодирование категорий.


In [ ]:
try:
    import kagglehub

    path = kagglehub.dataset_download("c/titanic")
    dataset_path = Path(path)
    csv_files = list(dataset_path.glob("*.csv"))
    train_file = next((f for f in csv_files if "train" in f.name.lower()), csv_files[0])
    df = pd.read_csv(train_file)
    print(f"Датасет загружен: {train_file.name}")
except Exception as e:
    print(f"Загрузка через kagglehub: {e}")
    url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    df = pd.read_csv(url)
    print("Датасет загружен через альтернативный источник")

print(f"\nРазмерность: {df.shape}")
df.head(10)


In [ ]:
df_clean = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"], errors="ignore")

numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df_clean.select_dtypes(include=["object"]).columns.tolist()

imputer_num = SimpleImputer(strategy="median")
df_clean[numeric_cols] = imputer_num.fit_transform(df_clean[numeric_cols])

for col in categorical_cols:
    mode_vals = df_clean[col].mode()
    fill_val = mode_vals.iloc[0] if len(mode_vals) > 0 else "Unknown"
    df_clean[col] = df_clean[col].fillna(fill_val)

df_encoded = pd.get_dummies(df_clean, columns=categorical_cols, drop_first=True)

X = df_encoded.drop(columns=["Survived"])
y = df_encoded["Survived"]

feature_names = X.columns.tolist()

print("Пропуски после обработки:", X.isnull().sum().sum())
print(f"Признаков: {X.shape[1]}, объектов: {X.shape[0]}")
X.head()


## 4. Разделение на обучающую и тестовую выборки

Стратификация по целевому признаку сохраняет доли классов.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Обучающая выборка: {X_train.shape[0]} объектов")
print(f"Тестовая выборка: {X_test.shape[0]} объектов")
print(f"Классы в train: {dict(y_train.value_counts())}")
print(f"Классы в test: {dict(y_test.value_counts())}")


## 5. Обучение ансамблевых моделей

| Модель | Тип |
|--------|-----|
| **RandomForestClassifier** | бэггинг деревьев с bootstrap по объектам и случайным подмножеством признаков |
| **ExtraTreesClassifier** | те же идеи бэггинга + более случайные пороги расщепления («сверхслучайные» деревья) |
| **AdaBoostClassifier** | последовательное усиление слабых классификаторов |
| **GradientBoostingClassifier** | градиентный бустинг Фридмана над деревьями |

Гиперпараметры зафиксированы одинаковым `random_state` для воспроизводимости.


In [ ]:
RANDOM_STATE = 42
N_EST_BAGGING = 300

models = {
    "Случайный лес (Random Forest)": RandomForestClassifier(
        n_estimators=N_EST_BAGGING,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "Сверхслучайные деревья (Extra Trees)": ExtraTreesClassifier(
        n_estimators=N_EST_BAGGING,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "AdaBoost": AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE),
        n_estimators=200,
        learning_rate=0.8,
        random_state=RANDOM_STATE,
        algorithm="SAMME",
    ),
    "Градиентный бустинг (GBM)": GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=3,
        random_state=RANDOM_STATE,
    ),
}

fitted = {}
predictions = {}

for name, est in models.items():
    est.fit(X_train, y_train)
    predictions[name] = est.predict(X_test)
    fitted[name] = est
    print(f"Обучено: {name}")


## 6. Метрика качества и сравнение моделей

В соответствии с заданием используется **одна** метрика — **accuracy** (доля верных ответов на тестовой выборке).

Для бинарной классификации с умеренным дисбалансом классов это простая и интерпретируемая метрика; при сильном перекосе классов дополнительно имеет смысл смотреть F1 или ROC-AUC.


In [ ]:
rows = []
for name in models:
    acc = accuracy_score(y_test, predictions[name])
    rows.append({"Модель": name, "Accuracy": acc})

metrics_df = pd.DataFrame(rows).sort_values("Accuracy", ascending=False).reset_index(drop=True)

print("Сравнение ансамблей по Accuracy на тесте")
print("=" * 55)
print(metrics_df.to_string(index=False))
print(f"\nЛучшая модель: {metrics_df.iloc[0]['Модель']} ({metrics_df.iloc[0]['Accuracy']:.4f})")


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
order = metrics_df.sort_values("Accuracy", ascending=True)
colors = sns.color_palette("viridis", n_colors=len(order))
ax.barh(order["Модель"], order["Accuracy"], color=colors, alpha=0.9)
ax.set_xlabel("Accuracy")
ax.set_title("Сравнение ансамблевых моделей")
ax.set_xlim(0, 1.02)
for i, v in enumerate(order["Accuracy"]):
    ax.text(v + 0.01, i, f"{v:.3f}", va="center", fontsize=10)
plt.tight_layout()
plt.show()


## 7. Выводы

1. Данные Titanic предобработаны: заполнены пропуски, категории закодированы.
2. Выборка разделена функцией `train_test_split` со стратификацией.
3. Обучены две модели семейства бэггинга (**случайный лес** и **Extra Trees**), **AdaBoost** и **градиентный бустинг**.
4. Качество на тесте оценено метрикой **accuracy**; по столбчатой диаграмме видно соотношение моделей.

При необходимости качество можно улучшить подбором `n_estimators`, глубины деревьев и скорости обучения у бустингов.
